# Πρόβλημα 1: Ομαδοποίηση κειμένων


## Βάλτε τα στοιχεία σας!

Πριν προχωρήσετε στην άσκηση συμπληρώστε το username και το password σας που σας έχει δοθεί και τρέξτε αυτό το κελί. Αυτές οι πληροφορίες είναι απαραίτητες για να σας αναγνωρίσει ο server και να βαθμολογήσει τις λύσεις σας.

- **Ο τελικός σας βαθμός καθορίζεται μόνο από το ΤΕΛΕΥΤΑΙΟ ΜΟΝΤΕΛΟ ΠΟΥ ΥΠΟΒΛΗΘΗΚΕ.**
- **Βεβαιωθείτε ότι χρησιμοποιείτε τα σωστά διαπιστευτήρια, διαφορετικά η λύση σας δεν θα βαθμολογηθεί.**

Εάν η ταυτοποίηση αποτύχει, θα δείτε ένα μήνυμα σφάλματος κάτω από την κλήση του `submit_for_grading()` στο τελευταίο κελί

In [1]:
import requests
import os

SERVER_URL = "http://195.251.252.26:5000"

submission_data = {
    'username': 'contestant15',
    'password': 'dV3^m4FbX',
    'exercise': 'lab1_clustering'
}

# The student must type the local name of this notebook file here if changed
NOTEBOOK_FILE = "/content/drive/MyDrive/Colab Notebooks/01_greek_proverb_clustering.ipynb"
## NOTEBOOK_FILE = "01_greek_proverb_clustering.ipynb"

FILE_CONFIG = {
    "lab1_clustering": {
        "filename": "predictions.csv",
        "mime": "text/csv"
    }
}

Zητείται να πραγματοποιήσετε ομαδοποίηση (clustering) των κειμένων που σας δίνονται (ελληνικές παροιμίες), εξάγοντας χαρακτηριστικά, βρίσκοντας τον βέλτιστο αριθμό ομάδων (clusters) και εκπαιδεύοντας τον αλγόριθμο που επιλέξατε. Οι προβλέψεις του μοντέλου σας θα αξιολογηθούν σε ένα άγνωστο σύνολο δεδομένων (unseen dataset) για το οποίο διαθέτουμε επισημειώσεις για αναφορά (ground truth annotations). Προτείνουμε τη χρήση των ακόλουθων δύο μετρικών αξιολόγησης, αλλά σημειώστε ότι η τελική αξιολόγηση βασίζεται στον μέσο όρο του NMI με τους αξιολογητές.

### Ο δείκτης Silhouette

Για κάθε σημείο $x_i \in X$, ο δείκτης Silhouette ορίζεται ως εξής:

* Η συνοχή $a(x_i)$ μετράει πόσο παρόμοιο είναι το $x_i$ με την ομάδα στην οποία έχει ανατεθεί, υπολογίζοντας τη μέση απόσταση μεταξύ του $x_i$ και των υπόλοιπων σημείων της ίδιας ομάδας.

* Ο διαχωρισμός $b(x_i)$ είναι η μέση απόσταση μεταξύ του $x_i$ και των στοιχείων της πλησιέστερης ομάδας (στην οποία το $x_i$ δεν ανήκει).

* Η συνοχή και ο διαχωρισμός συνδυάζονται για να δώσουν τον τελικό δείκτη (καλύτερες είναι οι υψηλότερες τιμές):

$$\frac{b(x_i) - a(x_i)}{\max( \{a(x_i), b(x_i)\} )}$$

### ΝΜΙ: Normalised Mutual Information

* Η αμοιβαία πληροφορία (Mutual Information) μετρά την ομοιότητα μεταξύ δύο διαφορετικών αναθέσεων (cluster labelings) για τα ίδια $N$ σημεία δεδομένων.
* Έστω δύο ομαδοποιήσεις, $U$ και $V$, όπου $K^U$ και $K^V$ είναι οι μοναδικές κλάσεις που ανατέθηκαν από τους επισημειωτές/αλγορίθμους $A_U$ και $A_V$ αντίστοιχα. Το $MI(U, V)$ βασίζεται στα κοινά σημεία $|U_i \cap V_j|$ μεταξύ οποιωνδήποτε δύο ομάδων $U_i$ και $V_j$, και μετρά πόσο η γνώση της μίας μειώνει την αβεβαιότητα για την άλλη. Αν οι ομαδοποιήσεις είναι ανεξάρτητες, η γνώση της μίας δεν παρέχει καμία πληροφορία για την άλλη (δηλαδή, έχουμε χαμηλό $MI$). Αν είναι πλήρως ευθυγραμμισμένες, η γνώση της μίας μας επιτρέπει να προσδιορίσουμε ακριβώς την άλλη (δηλαδή, έχουμε υψηλό $MI$).
* Το NMI μοντελοποιεί την ίδια έννοια, αλλά είναι κανονικοποιημένο στο διάστημα $[0, 1]$.

### Μοντέλο αναφοράς: Ο αλγόριθμος K-Means

***
__Είσοδος__: X παρατηρήσεις, το πλήθος K των ομάδων <br>
__Έξοδος__: K ομάδες <br>
__Αποτέλεσμα__: Ένα μοντέλο που ταξινομεί οποιοδήποτε $x$ σε μία από τις $K$ ομάδες <br>
1. &emsp; Επιλέξτε K τυχαία παραδείγματα για να αρχικοποιήσετε τα κέντρα $C$.
2. &emsp; Αναθέστε κάθε $x \in X$ στο πλησιέστερο κέντρο $c \in C$.
3. &emsp; Υπολογίστε τη μέση τιμή ανά ομάδα (cluster).
4. &emsp; Επαναλάβετε τα βήματα 2 και 3 μέχρι να υπάρξει σύγκλιση.
***

### Τα επόμενα βήματα

* __Επιλέξτε τον αλγόριθμο!__
    * Ο K-Means απαιτεί να ορίσετε την παράμετρο $K$.
    * Μπορείτε να πειραματιστείτε και με άλλους αλγορίθμους, που πιθανώς οδηγούν σε καλύτερα αποτελέσματα.
* __Επιλέξτε την κατάλληλη αναπαράσταση των κειμένων!__
    * Το TF-IDF είναι ένας γρήγορος τρόπος αναπαράστασης κάθε κειμένου, βασισμένος στις συχνότητες των όρων (term frequencies). Θα πρέπει να το παραμετροποιήσετε κατάλληλα.
    * Είστε ελεύθεροι να πειραματιστείτε και με άλλες στρατηγικές, που ενδεχομένως οδηγούν σε καλύτερα αποτελέσματα.

# Ξεκινήστε εδώ!

## Φόρτωση των δεδομένων

In [2]:
import gdown
import os

folder_id = '1poyoIEkVaTyBcxA4Hye7b0kQBnp_54sg'
url = f'https://drive.google.com/drive/folders/{folder_id}?usp=sharing'
output_directory = '.'

if not os.path.exists(output_directory):
    os.makedirs(output_directory)
gdown.download_folder(url, output=output_directory, quiet=False, remaining_ok=True)
print(f"Successfully downloaded data to {output_directory}")

Retrieving folder contents


Processing file 1_uJHcDuZr-B1G4RDCX1qC69WhMRQUWry cluster_non_annotations.csv
Processing file 1Pi6air6yJMreOMGt-H8ZTMDs0kradQoz proverbs.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1_uJHcDuZr-B1G4RDCX1qC69WhMRQUWry
To: /content/cluster_non_annotations.csv
100%|██████████| 8.07k/8.07k [00:00<00:00, 15.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Pi6air6yJMreOMGt-H8ZTMDs0kradQoz
To: /content/proverbs.csv
100%|██████████| 876k/876k [00:00<00:00, 9.38MB/s]

Successfully downloaded data to .



Download completed


In [3]:
import pandas as pd
import numpy as np
np.random.seed(2026)
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

seen = pd.read_csv('proverbs.csv')
seen.sample()

,text,label
8908,Όποιος αφκρέται τα δκα 'τ' ακούει,NaN


## Επιλογή του αλγορίθμου
Αλλάξτε το ελεύθερα, αλλάζοντας το `k`, τον αλγόριθμο, την αρχικοποίηση του `TFIDF`, την στρατηγική αναπαράστασης των κειμένων.

In [4]:
from tqdm.notebook import tqdm
from sklearn.metrics import silhouette_score, silhouette_samples
import seaborn as sns
sns.set_style('whitegrid')

k = 17 # select a better one, it must be 5 <= k <= 17
model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000)), # add parameters
    ('kmeans', KMeans(n_clusters=k, init='random', random_state=2026)) # keep this algorithm or not
])
model.fit(seen.text)
X = model['tfidf'].transform(seen.text)
labels = model.predict(seen.text)

## Υπολογισμός της Λύσης

Υποθέτουμε ότι ο αλγόριθμος που έχει επιλεγεί ονομάζεται `model` και το αρχείο που σώζεται ονομάζεται `predictions.csv`.

In [5]:
unseen = pd.read_csv('cluster_non_annotations.csv')
unseen['label'] = model.predict(unseen.text.values)
unseen.to_csv('predictions.csv', index=False)
unseen.sample()

,text,label
82,"Όπου ακούσης μέγα τρύγος, παίρνε μικρό καλάθι",1


## Υποβολή της Λύσης και Αξιολόγηση

Το παρακάτω κελί υποβάλει την λύση σας (και το notebook που την υπολόγισε) στο σύστημα αξιολόγησης. Ως ανατροφοδότηση παρέχεται ο δείκτης ΝΜΙ της ομαδοποίησης σας για ένα μικρό δημόσιο σύνολο (περίπου 100) ελληνικών παροιμιών και η αντίστοιχη βαθμολογία.

Η τελική βαθμολογία προκύπτει από την **τιμή του NMI** σε ένα σημαντικά μεγαλύτερο ιδιωτικό σύνολο (μερικών χιλιάδων) ελληνικών παροιμιών. **Αν το πλήθος k των ομάδων που χρησιμοποιήσατε είναι μεταξύ 5 και 17**, η τελική βαθμολογία θα προκύψει από την τιμή του ΝΜΙ (ενδεικτικά αναφέρουμε πως **τιμές NMI που ξεπερνούν το 0.5** θα έχουν υψηλή βαθμολογία και **τιμές NMI μικρότερες του 0.15** θα έχουν μηδενική βαθμολογία). **Αν το πλήθος των ομάδων k είναι μικρότερο του 5 ή μεγαλύτερο του 17, η τελική βαθμολογία θα είναι μηδενική**.

In [6]:
#########################################
####    DO NOT CHANGE THIS CELL      ####
#########################################
from google.colab import drive
drive.mount('/content/drive')

def submit_for_grading():
    """Submits the generated predictions/weights for automated grading."""
    username = submission_data.get('username')
    exercise = submission_data.get('exercise')
    output_file = f"{username}_{exercise}_final_score.csv"
    endpoint = f"{SERVER_URL}/grade"

    config = FILE_CONFIG.get(exercise)
    if not config:
        print(f" Error: Unknown exercise '{exercise}'.")
        return

    input_file = config["filename"]
    mime_type = config["mime"]

    if not os.path.exists(input_file):
        print(f" Error: Could not find '{input_file}'. Have you generated your submission file yet?")
        return

    print(f"Connecting to {endpoint}...")
    print(f"Submitting {input_file} for {exercise}...")

    try:
        with open(input_file, 'rb') as f:
            files = {'file': (input_file, f, mime_type)}
            response = requests.post(endpoint, files=files, data=submission_data)

        if response.status_code == 200:
            with open(output_file, 'wb') as f_out:
                f_out.write(response.content)
            print(f" Grading Success! Your score report has been saved to '{output_file}'.")
        else:
            print_server_error(response)

    except requests.exceptions.ConnectionError:
        print(" Failed to connect to the grading server. Check your network or VPN.")


def submit_notebook():
    """Uploads the Jupyter Notebook to the grading server."""
    endpoint = f"{SERVER_URL}/submit_notebook"

    if not os.path.exists(NOTEBOOK_FILE):
        print(f" Error: Could not find notebook '{NOTEBOOK_FILE}'. Check the filename.")
        return

    print(f"\nArchiving notebook {NOTEBOOK_FILE} to the server...")

    try:
        with open(NOTEBOOK_FILE, 'rb') as f:
            files = {'file': (NOTEBOOK_FILE, f, 'application/x-ipynb+json')}
            response = requests.post(endpoint, files=files, data=submission_data)

        if response.status_code == 200:
            print(f" Notebook successfully archived on the server!")
        else:
            print_server_error(response)

    except requests.exceptions.ConnectionError:
        print(" Failed to connect to the server. Check your network or VPN.")


def print_server_error(response):
    """Helper function to parse and display server errors cleanly."""
    try:
        error_data = response.json()
        print(f"\n SERVER ERROR {response.status_code} ")
        print(f"Summary: {error_data.get('error', 'Unknown Error')}")
        if 'details' in error_data:
            print(f"Details:\n{error_data['details']}")
    except:
        print(f"\n Server Error {response.status_code}: {response.text}")


if __name__ == "__main__":
    # 1. Submit the predictions for an automated score
    submit_for_grading()

    # 2. Upload the notebook code to the TA archive
    submit_notebook()

Mounted at /content/drive
Connecting to http://195.251.252.26:5000/grade...
Submitting predictions.csv for lab1_clustering...
 Failed to connect to the grading server. Check your network or VPN.

Archiving notebook /content/drive/MyDrive/Colab Notebooks/01_greek_proverb_clustering.ipynb to the server...
 Failed to connect to the server. Check your network or VPN.
